# Automatische Textzusammenfassung mit KI
### Vortrainierte Sprachmodelle mit HuggingFace Transformers

---

## Lernziele

Nach dieser Einheit könnt ihr:
- erklären, was ein **vortrainiertes Sprachmodell** ist und wie es genutzt wird
- die Rolle von **Tokenizer** und **Pipeline** in einer NLP-Anwendung beschreiben
- ein fertiges KI-Modell in Python einbinden und anwenden
- die **Stärken und Grenzen** automatischer Textzusammenfassung einschätzen

---

## Einstieg: Was kann KI mit Sprache anfangen?

Moderne KI-Modelle können Texte nicht nur erzeugen, sondern auch **verstehen und verarbeiten** – zum Beispiel zusammenfassen, übersetzen oder klassifizieren. Solche Aufgaben nennt man **Natural Language Processing (NLP)**.

In dieser Einheit nutzt ihr ein vortrainiertes Modell von [HuggingFace](https://huggingface.co), um Texte automatisch zusammenzufassen. Ihr werdet sehen, was das Modell leistet – und wo seine Grenzen liegen.

> **Hinweis:** Das verwendete Modell (`distilbart-cnn-12-6`) wurde auf englischen Nachrichtentexten trainiert. Euer Eingabetext muss daher **auf Englisch** sein, damit die Zusammenfassung sinnvoll ausfällt.

---
## Schritt 1: Bibliotheken installieren

Wir benötigen zwei Bibliotheken:
- **PyTorch** (`torch`): ein Framework für maschinelles Lernen, das die Berechnungen im Hintergrund übernimmt
- **Transformers** (von HuggingFace): stellt vortrainierte Modelle und einfache Schnittstellen (Pipelines) bereit

Die Zellen unten müssen nur **einmalig** ausgeführt werden.

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install "transformers<5.0.0"

---
## Schritt 2: Hintergrundwissen – Was passiert hier eigentlich?

Bevor ihr den Code ausführt, lest die folgenden Erklärungen durch:

### Vortrainiertes Modell
Ein vortrainiertes Modell wurde bereits auf riesigen Textmengen trainiert (hier: Millionen englischer Nachrichtenartikel). Wir laden dieses fertige Modell herunter und nutzen es direkt – ohne es selbst trainieren zu müssen. Das spart enorm viel Rechenzeit und Daten.

### Tokenizer
Computer können keinen Text direkt verarbeiten. Der **Tokenizer** zerlegt den Eingabetext zunächst in kleine Einheiten (sog. *Tokens*) – das können Wörter, Wortteile oder Satzzeichen sein – und wandelt diese in Zahlen um, mit denen das Modell rechnen kann.

Beispiel: `"Artificial intelligence"` → `[0, 43788, 4454, 2]`

### Pipeline
Die `pipeline`-Funktion von HuggingFace verpackt Tokenizer und Modell in eine einfache Schnittstelle. Man gibt einen Text hinein und erhält das Ergebnis direkt zurück – ohne sich um die internen Schritte kümmern zu müssen.

---
## Schritt 3: Modell laden und Text zusammenfassen

Jetzt laden wir das Modell und führen die Zusammenfassung durch.

> **Achtung:** Der Download des Modells dauert beim ersten Ausführen einige Minuten (ca. 1 GB). Danach wird es lokal zwischengespeichert.

In [ ]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

# --- Eingabe ---
# Gebt hier euren englischen Text ein (mind. 3-4 Saetze empfohlen)
text = input("Euer englischer Text: ")

# --- Modell und Tokenizer laden ---
# DistilBART ist eine schnellere, kompaktere Version von BART
# Trainiert auf englischen CNN- und Daily-Mail-Artikeln
model_name = "sshleifer/distilbart-cnn-12-6"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)   # laedt die Modellgewichte (~1 GB)
tokenizer = AutoTokenizer.from_pretrained(model_name)        # laedt den passenden Tokenizer

# --- Pipeline erstellen ---
# device=-1 bedeutet: Berechnung auf der CPU (kein Grafikprozessor noetig)
pipe = pipeline("summarization", model=model, tokenizer=tokenizer, device=-1)

# --- Parameter fuer die Zusammenfassung ---
# min_length: Mindestlaenge der Zusammenfassung in Tokens
# max_length: Maximale Laenge der Zusammenfassung in Tokens
# (Experimentiert mit diesen Werten – Aufgabe 2!)
min_length = 30
max_length = 130

# --- Zusammenfassung erzeugen ---
ergebnis = pipe(text, truncation=True, min_length=min_length, max_length=max_length)

print("\n--- Zusammenfassung ---")
print(ergebnis[0]['summary_text'])

---
## Aufgaben

### Aufgabe 1 – Erste Erkundung
Gebt einen englischen Nachrichtentext ein (z.B. von [BBC News](https://www.bbc.com/news) oder [Reuters](https://www.reuters.com), mindestens 150 Wörter). Führt die Zusammenfassung aus.
- Wie gut trifft die Zusammenfassung den Kerninhalt des Textes?
- Welche Informationen fehlen? Welche sind enthalten?

### Aufgabe 2 – Parameter verändern
Ändert die Werte von `min_length` und `max_length` im Code und führt die Zelle erneut aus:
- Versuch 1: `min_length=10`, `max_length=50`
- Versuch 2: `min_length=80`, `max_length=200`

Was verändert sich an der Ausgabe? Was passiert, wenn `min_length` größer als der Eingabetext ist?

### Aufgabe 3 – Grenzen des Modells
Testet das Modell bewusst in Extremsituationen. Gebt nacheinander folgende Texte ein und notiert die Ergebnisse:
- a) Einen sehr kurzen Text (1–2 Sätze)
- b) Einen Text mit einer klaren, einseitigen Meinung oder einer Falschaussage
- c) Einen deutschen Text

Was beobachtet ihr? Was lässt sich daraus über die Zuverlässigkeit solcher Modelle schließen?

### Aufgabe 4 – Vertiefung (optional)
Recherchiert auf [huggingface.co/models](https://huggingface.co/models) nach einem Modell, das **Deutsche Texte** zusammenfassen kann. Ändert `model_name` im Code entsprechend und vergleicht die Ergebnisse mit dem englischen Modell.

---
## Reflexion und Diskussion

Diskutiert die folgenden Fragen in der Klasse oder haltet eure Überlegungen schriftlich fest:

1. **Nützlichkeit:** In welchen Alltagssituationen könnte automatische Textzusammenfassung sinnvoll eingesetzt werden? Wo wäre Vorsicht geboten?

2. **Bias und Trainingsdaten:** Das Modell wurde auf Nachrichtenartikeln von CNN und Daily Mail trainiert. Welche Auswirkungen könnte das auf die Qualität und Neutralität der Zusammenfassungen haben?

3. **Verantwortung:** Wer trägt Verantwortung, wenn eine automatisch erzeugte Zusammenfassung wichtige Informationen weglässt oder verfälscht und jemand daraufhin eine Fehlentscheidung trifft?

4. **Datenschutz:** Was sollte man bedenken, bevor man vertrauliche oder persönliche Texte in ein KI-System eingibt?